# Barra多因子模型

Barra纯因子模型针对回答了一个问题：对于给定的因子，如何构建因子投资组合呢？常见的做法是，将所有个股在该因子上的因子暴露在截面上标准化；之后所有股票会按照因子的业务逻辑、根据因子暴露的数值从好到坏排列；最后，假设做多前 10% 或者 20% 的股票，做空后 10% 或者 20% 的股票，以此来构建一个零额投资的投资组合，它就是该因子的投资组合。这个做法在业界非常流行，但它也存在两个缺陷：

1. 无法保证该投资组合对该因子的暴露为 1。
2. 无法保证该投资组合对其他因子的因子暴露为 0。

Barra纯因子模型（pure factor model），它能够保证在截面上构建因子投资组合时，每个因子的投资组合对目标因子有 1 个单位的暴露，而对其他因子的暴露为 0。

>  严谨的说，根据因子的性质不同（即国家因子、行业因子、风格因子），因子的投资组合在其他因子上是否完全为 0 单位暴露略有差异（下文会具体说明）。但这不影响我们从广义上说“Barra 的模型中，因子的投资组合对目标因子有 1 个单位的暴露，对其他因子没有暴露”。这就是纯因子模型中“纯”字的含义。

## 理论介绍

### 模型形式CNE5

CNE5 是 Barra 的最新一代面向中国股票市场的多因子模型。该模型考虑了一个国家因子、多个行业因子以及多个风格因子。假设市场中共有 $N$ 支股票，$P$ 个行业，以及 $Q$ 个风格因子。在任意给定时间点，该模型使用因子暴露和下期的个股收益率构建截面回归（cross-sectional regression）。其中 $r_n$ 是第 n 支股票的收益率，$r_f$ 是无风险收益率。$X_n^{I_p}$ 是股票 n 在行业 $I_p$ 的暴露，如果假设一个公司只能属于一个行业，那么 $X_n^{I_p}$ 的取值为 0（代表该股票不属于这个行业）或者 1（代表该股票属于这个行业）。$X_n^{S_q}$ 是股票 n 在风格因子 $S_q$ 的暴露，它的取值经过了某种标准化（标准化的方法会在下文说明）。$u_n$ 为股票 n 的超额收益中无法被因子解释的部分，因此也被称为该股票的特异性收益。$f_C$ 为国家因子的因子收益率（所有股票在国家因子上的暴露都是1）；$f_{I_p}$ 为行业 $I_p$ 因子的因子收益率；$f_{S_q}$ 为风格因子 $S_q$ 的因子收益率。

$$

\begin{equation*}
\begin{bmatrix}
r_1 - r_f \\
r_2 - r_f \\
\vdots \\
r_N - r_f
\end{bmatrix}
=
\begin{bmatrix}
1 \\
1 \\
\vdots \\
1
\end{bmatrix}
f_c +
\begin{bmatrix}
X_1^{l_1} \\
X_2^{l_1} \\
\vdots \\
X_N^{l_1}
\end{bmatrix}
f_1 + \cdots +
\begin{bmatrix}
X_1^{l_p} \\
X_2^{l_p} \\
\vdots \\
X_N^{l_p}
\end{bmatrix}
f_{l_p} +
\begin{bmatrix}
X_1^{s_1} \\
X_2^{s_1} \\
\vdots \\
X_N^{s_1}
\end{bmatrix}
f_{s_1} + \cdots +
\begin{bmatrix}
X_1^{s_Q} \\
X_2^{s_Q} \\
\vdots \\
X_N^{s_Q}
\end{bmatrix}
f_{s_Q} +
\begin{bmatrix}
u_1 \\
u_2 \\
\vdots \\
u_N
\end{bmatrix}
\end{equation*}

$$

上式就是 CNE5 多因子模型。在这个模型中，国家因子的因子暴露和 P 个行业的因子暴露之间存在共线性。具体来说，国家因子的因子暴露向量可以表达为 P 个行业因子因子暴露向量的线性组合。这会造成上式的解不唯一。

设截面回归（忽略风格先不影响结论）：

$$
r_n-r_f = f_C\cdot 1 + \sum_{p=1}^P X^{I_p}_n f_{I_p} + u_n
$$

如果“一家公司只能属于一个行业”，则对每只股票 $n$ ，行业哑变量满足：

- $ X^{I_p}_n \in \{0,1\} $
- 且 $ \sum_{p=1}^P X^{I_p}_n = 1 $ （每只股票恰好落在一个行业）

把所有股票的行业暴露写成矩阵 $ $X_I\in\mathbb{R}^{N\times P} $ ，国家因子暴露向量是 $ \mathbf{1}\in\mathbb{R}^N $ 。那么有一个关键恒等式：

$$
\mathbf{1} = X_I \mathbf{1}_P
$$

其中 $ \mathbf{1}_P $ 是长度为 $ P $ 的全 1 向量。含义就是：**“国家因子那一列”恰好等于所有行业哑变量列的和**。

因此回归设计矩阵中，“国家列”与“行业列”线性相关，导致设计矩阵不满秩。

### 国家因子的本质

国家因子投资组合的实质是按流通市值为权重的市场组合。由前文所述，$s_n$ 是股票 $n$ 的流通市值权重。将 ${s_n}, n = 1, …, N$ 这一组权重带入到 CNE5 的因子模型中可以得到如下关系。其中左侧就是市场收益 $r_M$ ，右侧是使用国家因子、行业因子、风格因子、以及个股特异性收益率对 $r_M$ 的分解。

$$
\begin{aligned}
r_M &= f_C + \sum_{n=1}^N \sum_{p=1}^P s_n X_n^{I_p} f_{I_p} + \sum_{n=1}^N \sum_{q=1}^Q s_n X_n^{S_q} f_{S_q} + \sum_{n=1}^N s_n u_n \\
&= f_C + \sum_{p=1}^P \left( \sum_{n=1}^N s_n X_n^{I_p} \right) f_{I_p} + \sum_{q=1}^Q \left( \sum_{n=1}^N s_n X_n^{S_q} \right) f_{S_q} + \sum_{n=1}^N s_n u_n \\
&= f_C + \sum_{p=1}^P s_{I_p} f_{I_p} + \sum_{q=1}^Q 0 \times f_{S_q} + \sum_{n=1}^N s_n u_n \\
&= f_C + 0 + 0 + \sum_{n=1}^N s_n u_n \\
&= f_C + \sum_{n=1}^N s_n u_n \\
&\approx f_C
\end{aligned}
$$

上式中最后一项是所有股票特异性收益的和，由于它的值非常小（接近 0），因此在推导的最后一步被忽略了。推导中的核心在于倒数第三步中的中间两项如何变为 0。对于第一个 0，它用到了行业因子收益率按行业市值加权为 0 以排除行业和国家因子之间的共线性这个约束条件。对于第二个 0，它是根据风格因子是使用流通市值权重来标准化这个定义来的。由此可见，在 CNE5 模型的定义下，f_C 这个国家因子收益率确实近似的代表了市场组合的收益率，因此国家因子的组合就（近似地）是市场组合。在新版多因子模型中增加这一项是非常必要的。

事实上，在对 CNE5 进行截面回归求解后可以发现，国家因子的投资组合中，个股 $n$ 的权重 $ω_{Cn}$ 非常接近它的流通市值权重 $s_n$ 。

## 模型实现

### 参数设定

- BEGIN: 计算开始日
- END: 计算截止日
- TARGET_PATH: 用于计算未来收益的价格，参见因子路径
- HORIZON: 使用的未来收益周期
- WINDOW: 用于IC和合成风格因子时使用的时间窗口
- WEIGHT_PATH: 用于Barra多因子回归、因子市场中性化的权重因子路径
- DATASET_PATH: 因子数据库路径
- RETURN_TABLE: Barra回归的因子收益存放在因子数据库中的表名

In [ ]:
from pathlib import Path
from factool import DuckPQSource
from example.barra import (
    ICComposerConfig,
    BarraConfig,
    BarraComposer,
    load_industry_dummies,
)

BEGIN = "2025-01-01"
END = "2025-12-31"
HORIZON = 5
WINDOW = 252
TARGET_PATH = "target/open_post"
WEIGHT_PATH = "barra_size/mcap_float_a"
DATASET_PATH = "data"
RETURN_TABLE = f"barra_returns_h{HORIZON}"

fs = DuckPQSource(Path(DATASET_PATH))
fs.register(RETURN_TABLE)
fs.register("quotes_day")
fs.register("instruments_info")
fs.register("industry_mapping")

### Barra市值因子

In [ ]:
size_config = ICComposerConfig(
    factor_paths=[
        "barra_size/mcap_float_a",
        "barra_size/ln_mcap_float_a",
        "barra_size/ln_mcap_float_a_cu",
        "barra_size/non_linear_size",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra价值因子

In [ ]:
value_config = ICComposerConfig(
    factor_paths=[
        "barra_value/total_equity_mrq_to_mktcap",
        "barra_value/total_equity_ttm_to_mktcap",
        "barra_value/operating_revenue_mrq_to_mktcap",
        "barra_value/operating_revenue_ttm_to_mktcap",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra流动性因子

In [ ]:
liquidity_config = ICComposerConfig(
    factor_paths=[
        "barra_liquidity/barra_liquidity_turnover_63d",
        "barra_liquidity/barra_liquidity_amount_turnover",
        "barra_liquidity/barra_liquidity_amihud_illiquidity",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra杠杆因子

In [ ]:
leverage_config = ICComposerConfig(
    factor_paths=[
        "barra_leverage/barra_book_leverage",
        "barra_leverage/barra_market_leverage",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra波动率因子

In [ ]:
volatility_config = ICComposerConfig(
    factor_paths=[
        "barra_volatility/barra_vol_std_252",
        "barra_volatility/barra_beta_252",
        "barra_volatility/barra_residvol_252",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra动量因子

In [ ]:
momentum_config = ICComposerConfig(
    factor_paths=[
        "barra_momentum/barra_mom_st_63d",
        "barra_momentum/barra_mom_lt_126d_ex21d",
        "barra_momentum/barra_rev_st_21d",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra盈利因子

In [ ]:
profitability_config = ICComposerConfig(
    factor_paths=[
        "barra_profitability/ROA",
        "barra_profitability/ROE",
        "barra_profitability/asset_turnover",
        "barra_profitability/profit_margin",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra成长因子

In [ ]:
growth_config = ICComposerConfig(
    factor_paths=[
        "barra_growth/net_profit_qoq_gr_mean_20",
        "barra_growth/net_profit_qoq_gr_std_20",
        "barra_growth/net_profit_yoy_gr_mean_20",
        "barra_growth/net_profit_yoy_gr_std_20",
        "barra_growth/net_profit_accel_mean_20",
    ],
    begin=BEGIN,
    end=END,
    horizon=HORIZON,
    window=WINDOW,
    weight_path=WEIGHT_PATH,
)

### Barra合成器

In [ ]:
barra_config = BarraConfig(
    begin=BEGIN,
    end=END,
    factor_paths={
        "size": size_config,
        "value": value_config,
        "liquidity": liquidity_config,
        "leverage": leverage_config,
        "volatility": volatility_config,
        "momentum": momentum_config,
        "profitability": profitability_config,
        "growth": growth_config,
    },
    horizon=HORIZON,
    window=WINDOW,
    target_path=TARGET_PATH,
    weight_path=WEIGHT_PATH,
)

In [ ]:
industry = load_industry_dummies(source=fs, begin=BEGIN, end=END)
composer = BarraComposer(fs, barra_config, industry=industry)
factor_returns = composer.run(
    min_style_coverage=0.5,
    drop_industry_rule="max_cap",
    progress=True,
    progress_every=20,
)
style_exposure = composer.style_exposure.sort_index().reset_index()  # type: ignore
style_exposure["date"] = style_exposure["date"].dt.strftime("%Y-%m-%d")
fs.upsert("barra", style_exposure, keys=["date", "code"], partition_by=["date"])

factor_returns[0].index.name = "date"
factor_returns = factor_returns[0].reset_index()
fs.upsert(RETURN_TABLE, factor_returns, keys=["date"])